# 6. 作业题  

你可以根据自身能力选择基础、进阶作业完成，想多尝试也可以两部分都完成。每个作业需要提供详细的代码过程和结果文字总结。  

## 基础作业：改用不同的embedding模型获得向量嵌入  
详细描述：文本嵌入的质量决定了最后知识库检索的精准度，而embedding模型决定了文本嵌入的效果，因此，尝试使用不同的embedding模型获得向量嵌入，观察检索效果  

要求：  通过不同embedding模型获得文本向量嵌入，查看在同样的检索算法下，不同的检索效果  

提示：在modelscope上搜索不同的qwen3的embedding模型

In [ ]:
## 关键代码
# Requires transformers>=4.51.0
# Requires sentence-transformers>=2.7.0

from sentence_transformers import SentenceTransformer

# Load the model
model = SentenceTransformer("Qwen/Qwen3-Embedding-4B")

documents = [
    "The capital of China is Beijing.",
    "Gravity is a force that attracts two bodies towards each other. It gives weight to physical objects and is responsible for the movement of planets around the sun.",
]

document_embeddings = model.encode(documents)



（你的结论）

In [ ]:
# your code

## 进阶作业：改用不同的向量数据库进行向量存储  
详细描述：  向量数据库存储着文本向量，其如何压缩文本与生成索引关乎着检索的质量  

要求：  使用不同向量库，查看向量库所占用的资源  

提示：FAISS是大厂热门使用数据库，可以尝试使用FAISS来替代原有的向量库

In [ ]:
## 关键代码

import faiss
import numpy as np
import os

# 1. 准备数据
d = 128                     # 向量维度
nb = 10000                  # 向量数量
np.random.seed(42)
data = np.random.random((nb, d)).astype('float32')
query = np.random.random((1, d)).astype('float32')
k = 5                       # 检索前 k 个近邻

# 2. 创建两种索引
#  2.1 IndexFlatL2：精确暴力检索
index_flat = faiss.IndexFlatL2(d)

#  2.2 IndexIVFFlat：倒排文件 + 精确码本（需要先训练）
nlist = 100                 # 聚类中心数量
quantizer = faiss.IndexFlatL2(d)
index_ivf = faiss.IndexIVFFlat(quantizer, d, nlist, faiss.METRIC_L2)
index_ivf.train(data)       # 必须先用所有向量“训练”出聚类中心

# 3. 向索引中添加向量
index_flat.add(data)
index_ivf.add(data)

# 4. 执行检索
D_flat, I_flat = index_flat.search(query, k)
D_ivf, I_ivf = index_ivf.search(query, k)

print("=== FlatL2 检索结果 ===")
print("Distances:", D_flat)
print("Indices:  ", I_flat)

print("\n=== IVF 检索结果 ===")
print("Distances:", D_ivf)
print("Indices:  ", I_ivf)

# 5. 序列化索引并对比磁盘占用
faiss.write_index(index_flat, 'index_flat.faiss')
faiss.write_index(index_ivf,  'index_ivf.faiss')

size_flat = os.path.getsize('index_flat.faiss')
size_ivf  = os.path.getsize('index_ivf.faiss')
print(f"\nIndexFlatL2 文件大小：{size_flat/1024:.2f} KB")
print(f"IndexIVFFlat 文件大小：{size_ivf/1024:.2f} KB")

（你的结论）

In [ ]:
# your code